# Phase II — Model Training
## Goal: Train and evaluate two models, compare train/validation splits

Models:
- Model 1: Ridge Regression (linear baseline)
- Model 2: LightGBM (gradient boosting)

Splits to compare:
- 70/30 train-validation split
- 80/20 train-validation split

Metric: RMSLE (Root Mean Squared Logarithmic Error)

In [1]:
import numpy as np
import pandas as pd
import joblib
import os
import time
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from lightgbm import LGBMRegressor

print("All libraries loaded ✅")

All libraries loaded ✅


In [2]:
# Load Features and Target

print("Loading features and target...")

X = joblib.load('../data/processed/X_features.pkl')
y = joblib.load('../data/processed/y_target.pkl')

print(f"X shape : {X.shape}")
print(f"y shape : {y.shape}")
print(f"y range : {y.min():.3f} to {y.max():.3f}")
print(f"\nX type  : {type(X)}")

Loading features and target...
X shape : (1481661, 100009)
y shape : (1481661,)
y range : 1.386 to 7.606

X type  : <class 'scipy.sparse._csr.csr_matrix'>


In [4]:
# RMSLE Function

def rmsle(y_true, y_pred):
    """
    Calculate RMSLE.
    Since target is already log1p transformed,
    RMSLE = RMSE of log-transformed values.
    """
    return np.sqrt(mean_squared_error(y_true, y_pred))


def evaluate(model_name, y_true, y_pred):
    """Print evaluation metrics for a model."""
    score = rmsle(y_true, y_pred)
    rmse  = np.sqrt(mean_squared_error(y_true, y_pred))
    
    # Convert back to original price scale for interpretability
    y_true_price = np.expm1(y_true)
    y_pred_price = np.expm1(y_pred)
    mae_price    = np.mean(np.abs(y_true_price - y_pred_price))

    print(f"\n{'='*45}")
    print(f"  {model_name}")
    print(f"{'='*45}")
    print(f"  RMSLE          : {score:.4f}")
    print(f"  RMSE (log)     : {rmse:.4f}")
    print(f"  MAE (USD)      : ${mae_price:.2f}  ← real price scale")
    print(f"{'='*45}")
    
    return score

In [ ]:
# Split Comparison Function

def train_and_compare_splits(model, model_name, X, y):
    """
    Train model on 70/30 and 80/20 splits.
    Returns best split result.
    """
    results = {}

    for test_size, split_name in [(0.30, '70/30'), (0.20, '80/20')]:
        
        print(f"\nTraining {model_name} | Split: {split_name}")
        
        X_train, X_val, y_train, y_val = train_test_split(
            X, y,
            test_size=test_size,
            random_state=42    #   for reproducibility 
        )
        
        print(f"  Train size : {X_train.shape[0]:,}")
        print(f"  Val size   : {X_val.shape[0]:,}")
        
        start = time.time()
        model.fit(X_train, y_train)
        train_time = time.time() - start
        
        y_pred = model.predict(X_val)
        
        # Clip predictions — negative log_price
        y_pred = np.clip(y_pred, 0, None)
        
        score = evaluate(f"{model_name} | {split_name}", y_val, y_pred)
        print(f"  Train time : {train_time:.1f}s")
        
        results[split_name] = {
            'model'     : model,
            'score'     : score,
            'train_time': train_time,
            'X_train'   : X_train,
            'X_val'     : X_val,
            'y_train'   : y_train,
            'y_val'     : y_val,
            'y_pred'    : y_pred
        }
    
    # Better split বেছে নাও (lower RMSLE = better)
    best_split = min(results, key=lambda k: results[k]['score'])
    print(f"\n  ✅ Best split for {model_name}: {best_split}")
    
    return results, best_split

In [7]:
# Train Ridge Regression

print("=" * 50)
print("MODEL 1: RIDGE REGRESSION")
print("=" * 50)

ridge_model = Ridge(alpha=10, solver='sparse_cg', max_iter=100)

ridge_results, ridge_best_split = train_and_compare_splits(
    model=ridge_model,
    model_name="Ridge Regression",
    X=X,
    y=y
)

MODEL 1: RIDGE REGRESSION

Training Ridge Regression | Split: 70/30
  Train size : 1,037,162
  Val size   : 444,499

  Ridge Regression | 70/30
  RMSLE          : 0.6246
  RMSE (log)     : 0.6246
  MAE (USD)      : $14.17  ← real price scale
  Train time : 1.8s

Training Ridge Regression | Split: 80/20
  Train size : 1,185,328
  Val size   : 296,333

  Ridge Regression | 80/20
  RMSLE          : 0.6248
  RMSE (log)     : 0.6248
  MAE (USD)      : $14.16  ← real price scale
  Train time : 2.1s

  ✅ Best split for Ridge Regression: 70/30


In [8]:
# Train LightGBM

print("=" * 50)
print("MODEL 2: LIGHTGBM")
print("=" * 50)

lgbm_model = LGBMRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=63,
    max_depth=-1,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,          # use all CPU cores use
    verbose=-1          # training log suppress
)

lgbm_results, lgbm_best_split = train_and_compare_splits(
    model=lgbm_model,
    model_name="LightGBM",
    X=X,
    y=y
)

MODEL 2: LIGHTGBM

Training LightGBM | Split: 70/30
  Train size : 1,037,162
  Val size   : 444,499

  LightGBM | 70/30
  RMSLE          : 0.4660
  RMSE (log)     : 0.4660
  MAE (USD)      : $10.48  ← real price scale
  Train time : 603.6s

Training LightGBM | Split: 80/20
  Train size : 1,185,328
  Val size   : 296,333

  LightGBM | 80/20
  RMSLE          : 0.4650
  RMSE (log)     : 0.4650
  MAE (USD)      : $10.44  ← real price scale
  Train time : 677.4s

  ✅ Best split for LightGBM: 80/20


In [9]:
# Final Comparison Table

print("\n")
print("=" * 60)
print("  FINAL MODEL COMPARISON")
print("=" * 60)
print(f"  {'Model':<25} {'Split':<10} {'RMSLE':<10} {'Time'}")
print(f"  {'-'*55}")

for split in ['70/30', '80/20']:
    r_score = ridge_results[split]['score']
    r_time  = ridge_results[split]['train_time']
    l_score = lgbm_results[split]['score']
    l_time  = lgbm_results[split]['train_time']
    
    print(f"  {'Ridge Regression':<25} {split:<10} {r_score:<10.4f} {r_time:.1f}s")
    print(f"  {'LightGBM':<25} {split:<10} {l_score:<10.4f} {l_time:.1f}s")
    print(f"  {'-'*55}")

print(f"\n  Best Ridge  : {ridge_best_split}")
print(f"  Best LightGBM: {lgbm_best_split}")
print("=" * 60)



  FINAL MODEL COMPARISON
  Model                     Split      RMSLE      Time
  -------------------------------------------------------
  Ridge Regression          70/30      0.6246     1.8s
  LightGBM                  70/30      0.4660     603.6s
  -------------------------------------------------------
  Ridge Regression          80/20      0.6248     2.1s
  LightGBM                  80/20      0.4650     677.4s
  -------------------------------------------------------

  Best Ridge  : 70/30
  Best LightGBM: 80/20


In [10]:
# Best Models

os.makedirs('../data/processed', exist_ok=True)

# Best Ridge model save
best_ridge = ridge_results[ridge_best_split]['model']
joblib.dump(best_ridge, '../data/processed/ridge_model.pkl')

# Best LightGBM model save
best_lgbm = lgbm_results[lgbm_best_split]['model']
joblib.dump(best_lgbm, '../data/processed/lgbm_model.pkl')

print("Models saved:")
for f in ['ridge_model.pkl', 'lgbm_model.pkl']:
    size = os.path.getsize(f'../data/processed/{f}') / 1e6
    print(f"  {f:<30} {size:.1f} MB")

Models saved:
  ridge_model.pkl                0.8 MB
  lgbm_model.pkl                 10.2 MB


## Phase II Results Summary

| Model | Split | RMSLE | Notes |
|---|---|---|---|
| Ridge Regression | 70/30 | TBD | Linear baseline |
| Ridge Regression | 80/20 | TBD | Linear baseline |
| LightGBM | 70/30 | TBD | Gradient boosting |
| LightGBM | 80/20 | TBD | Gradient boosting |

### Key Observations:
- LightGBM expected to outperform Ridge (captures non-linear patterns)
- More training data (80/20) expected to improve both models
- Final model selected: TBD based on lowest RMSLE

### For Phase III Deployment:
- Best model saved to data/processed/
- All feature encoders saved from Phase I
- FastAPI will use same preprocessing pipeline